# Task 1: 数据预处理

本Notebook完成以下任务：
1. 解析SemEval-2014 ABSA数据集的XML文件
2. 构建Pipeline 1的数据集（方面类别检测 - 多标签分类）
3. 构建Pipeline 2的数据集（方面情感分析 - 4分类）
4. 保存为JSON/CSV格式供后续使用

## 1.1 导入必要的库

In [ ]:
import xml.etree.ElementTree as ET
import json
import pandas as pd
import os
from collections import defaultdict
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")

## 1.2 定义数据解析函数

In [ ]:
# 定义方面类别和情感标签
ASPECT_CATEGORIES = ['food', 'service', 'price', 'ambience', 'anecdotes/miscellaneous']
SENTIMENT_LABELS = ['positive', 'negative', 'neutral', 'conflict']

# 创建类别到索引的映射
ASPECT_TO_IDX = {cat: idx for idx, cat in enumerate(ASPECT_CATEGORIES)}
SENTIMENT_TO_IDX = {sent: idx for idx, sent in enumerate(SENTIMENT_LABELS)}

print("方面类别:", ASPECT_CATEGORIES)
print("情感标签:", SENTIMENT_LABELS)

In [ ]:
def parse_xml_file(xml_path: str) -> List[Dict]:
    """
    解析XML文件，提取句子、方面类别和情感信息
    
    Args:
        xml_path: XML文件路径
    
    Returns:
        解析后的数据列表
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    data = []
    for sentence in root.findall('sentence'):
        sentence_id = sentence.get('id')
        text_elem = sentence.find('text')
        
        if text_elem is None:
            continue
            
        text = text_elem.text
        if not text or not text.strip():
            continue
        
        record = {
            'id': sentence_id,
            'text': text.strip(),
            'aspect_terms': [],
            'aspect_categories': []
        }
        
        # 解析aspectTerms
        aspect_terms_elem = sentence.find('aspectTerms')
        if aspect_terms_elem is not None:
            for term in aspect_terms_elem.findall('aspectTerm'):
                record['aspect_terms'].append({
                    'term': term.get('term'),
                    'polarity': term.get('polarity'),
                    'from': term.get('from'),
                    'to': term.get('to')
                })
        
        # 解析aspectCategories
        aspect_categories_elem = sentence.find('aspectCategories')
        if aspect_categories_elem is not None:
            for category in aspect_categories_elem.findall('aspectCategory'):
                record['aspect_categories'].append({
                    'category': category.get('category'),
                    'polarity': category.get('polarity')
                })
        
        data.append(record)
    
    return data

print("✓ 解析函数定义完成")

## 1.3 解析所有XML文件

In [ ]:
# 定义文件路径
TRAIN_FILE = 'Restaurants train data.xml'
TRIAL_FILE = 'Restaurants trial data.xml'
TEST_PHASE_A_FILE = 'Restaurants_Test_Data_PhaseA.xml'
TEST_PHASE_B_FILE = 'Restaurants_Test_Data_phaseB.xml'

# 解析所有文件
print("正在解析训练集...")
train_data = parse_xml_file(TRAIN_FILE)
print(f"训练集: {len(train_data)} 条样本")

print("正在解析验证集...")
trial_data = parse_xml_file(TRIAL_FILE)
print(f"验证集: {len(trial_data)} 条样本")

print("正在解析测试集Phase A...")
test_phase_a_data = parse_xml_file(TEST_PHASE_A_FILE)
print(f"测试集Phase A: {len(test_phase_a_data)} 条样本")

print("正在解析测试集Phase B...")
test_phase_b_data = parse_xml_file(TEST_PHASE_B_FILE)
print(f"测试集Phase B: {len(test_phase_b_data)} 条样本")

## 1.4 数据统计分析

In [ ]:
def analyze_dataset(data: List[Dict], name: str):
    """分析数据集的统计信息"""
    print(f"\n===== {name} 统计 =====")
    print(f"样本总数: {len(data)}")
    
    # 统计有方面类别的样本数
    samples_with_categories = sum(1 for item in data if item['aspect_categories'])
    print(f"有方面类别的样本数: {samples_with_categories}")
    
    # 统计方面类别分布
    category_counts = defaultdict(int)
    sentiment_counts = defaultdict(int)
    
    for item in data:
        for cat in item['aspect_categories']:
            category_counts[cat['category']] += 1
            sentiment_counts[cat['polarity']] += 1
    
    print("\n方面类别分布:")
    for cat in ASPECT_CATEGORIES:
        print(f"  {cat}: {category_counts[cat]}")
    
    print("\n情感标签分布:")
    for sent in SENTIMENT_LABELS:
        print(f"  {sent}: {sentiment_counts[sent]}")

# 分析训练集
analyze_dataset(train_data, "训练集")

# 分析验证集
analyze_dataset(trial_data, "验证集")

## 1.5 构建Pipeline 1数据集：方面类别检测

In [ ]:
def build_aspect_detection_dataset(data: List[Dict]) -> pd.DataFrame:
    """
    构建方面类别检测数据集（多标签分类）
    
    输出格式：
    {
        "text": "评论文本",
        "labels": [1,0,1,0,0],  # [food, service, price, ambience, anecdotes]
        "label_food": 1,
        "label_service": 0,
        "label_price": 1,
        "label_ambience": 0,
        "label_anecdotes": 0
    }
    """
    records = []
    
    for item in data:
        text = item['text']
        categories = item['aspect_categories']
        
        # 初始化标签向量（5个类别）
        label_vector = [0] * len(ASPECT_CATEGORIES)
        
        # 设置存在的类别标签为1
        for cat in categories:
            cat_name = cat['category']
            if cat_name in ASPECT_TO_IDX:
                label_vector[ASPECT_TO_IDX[cat_name]] = 1
        
        record = {
            'text': text,
            'labels': label_vector,
        }
        
        # 添加单独的标签列
        for i, cat in enumerate(ASPECT_CATEGORIES):
            record[f'label_{cat.replace("/", "_")}'] = label_vector[i]
        
        records.append(record)
    
    return pd.DataFrame(records)

print("构建Pipeline 1数据集...")
train_aspect_df = build_aspect_detection_dataset(train_data)
trial_aspect_df = build_aspect_detection_dataset(trial_data)

print(f"训练集样本数: {len(train_aspect_df)}")
print(f"验证集样本数: {len(trial_aspect_df)}")
print("\n样本示例:")
print(train_aspect_df.head())

## 1.6 构建Pipeline 2数据集：方面情感分析

In [ ]:
def build_sentiment_dataset(data: List[Dict]) -> pd.DataFrame:
    """
    构建方面情感分析数据集（4分类）
    
    输出格式：
    {
        "text": "评论文本",
        "aspect": "food",
        "sentiment": "positive",
        "sentiment_id": 0  # positive:0, negative:1, neutral:2, conflict:3
    }
    """
    records = []
    
    for item in data:
        text = item['text']
        categories = item['aspect_categories']
        
        for cat in categories:
            cat_name = cat['category']
            sentiment = cat['polarity']
            
            # 跳过未标注的
            if sentiment not in SENTIMENT_TO_IDX:
                continue
            
            record = {
                'text': text,
                'aspect': cat_name,
                'sentiment': sentiment,
                'sentiment_id': SENTIMENT_TO_IDX[sentiment]
            }
            records.append(record)
    
    return pd.DataFrame(records)

print("构建Pipeline 2数据集...")
train_sentiment_df = build_sentiment_dataset(train_data)
trial_sentiment_df = build_sentiment_dataset(trial_data)

print(f"训练集样本数: {len(train_sentiment_df)}")
print(f"验证集样本数: {len(trial_sentiment_df)}")
print("\n样本示例:")
print(train_sentiment_df.head())

## 1.7 数据可视化分析

In [ ]:
# 方面类别分布可视化
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pipeline 1: 方面类别分布
aspect_cols = [f'label_{cat.replace("/", "_")}' for cat in ASPECT_CATEGORIES]
aspect_counts = train_aspect_df[aspect_cols].sum().values

axes[0].bar(ASPECT_CATEGORIES, aspect_counts, color='skyblue')
axes[0].set_title('Pipeline 1: 方面类别分布（训练集）')
axes[0].set_xlabel('方面类别')
axes[0].set_ylabel('样本数')
axes[0].tick_params(axis='x', rotation=45)

# Pipeline 2: 情感分布
sentiment_counts = train_sentiment_df['sentiment'].value_counts()
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', colors=['#90EE90', '#FFB6C1', '#87CEEB', '#FFD700'])
axes[1].set_title('Pipeline 2: 情感分布（训练集）')

plt.tight_layout()
plt.savefig('GroupXX_Dataset_files/data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 数据分布图已保存")

## 1.8 保存处理后的数据集

In [ ]:
# 创建输出目录
os.makedirs('GroupXX_Dataset_files', exist_ok=True)

# 保存Pipeline 1数据集（方面类别检测）
train_aspect_df.to_json('GroupXX_Dataset_files/train_aspect_detection.json', orient='records', force_ascii=False)
train_aspect_df.to_csv('GroupXX_Dataset_files/train_aspect_detection.csv', index=False)

trial_aspect_df.to_json('GroupXX_Dataset_files/trial_aspect_detection.json', orient='records', force_ascii=False)
trial_aspect_df.to_csv('GroupXX_Dataset_files/trial_aspect_detection.csv', index=False)

print("✓ Pipeline 1数据集已保存")
print(f"  - train_aspect_detection.json/csv: {len(train_aspect_df)}条")
print(f"  - trial_aspect_detection.json/csv: {len(trial_aspect_df)}条")

In [ ]:
# 保存Pipeline 2数据集（方面情感分析）
train_sentiment_df.to_json('GroupXX_Dataset_files/train_sentiment.json', orient='records', force_ascii=False)
train_sentiment_df.to_csv('GroupXX_Dataset_files/train_sentiment.csv', index=False)

trial_sentiment_df.to_json('GroupXX_Dataset_files/trial_sentiment.json', orient='records', force_ascii=False)
trial_sentiment_df.to_csv('GroupXX_Dataset_files/trial_sentiment.csv', index=False)

print("✓ Pipeline 2数据集已保存")
print(f"  - train_sentiment.json/csv: {len(train_sentiment_df)}条")
print(f"  - trial_sentiment.json/csv: {len(trial_sentiment_df)}条")

In [ ]:
# 保存原始解析的数据（包含所有信息）
with open('GroupXX_Dataset_files/train_data_raw.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open('GroupXX_Dataset_files/trial_data_raw.json', 'w', encoding='utf-8') as f:
    json.dump(trial_data, f, ensure_ascii=False, indent=2)

with open('GroupXX_Dataset_files/test_phase_a_data.json', 'w', encoding='utf-8') as f:
    json.dump(test_phase_a_data, f, ensure_ascii=False, indent=2)

with open('GroupXX_Dataset_files/test_phase_b_data.json', 'w', encoding='utf-8') as f:
    json.dump(test_phase_b_data, f, ensure_ascii=False, indent=2)

print("✓ 原始解析数据已保存")

## 1.9 数据摘要报告

In [ ]:
# 生成数据摘要
summary = f"""
# SemEval-2014 ABSA 数据集处理摘要

## 数据集统计

### 原始数据
- 训练集: {len(train_data)} 条评论
- 验证集: {len(trial_data)} 条评论
- 测试集Phase A: {len(test_phase_a_data)} 条评论（无标注）
- 测试集Phase B: {len(test_phase_b_data)} 条评论（部分标注）

### Pipeline 1: 方面类别检测（多标签分类）
- 训练集: {len(train_aspect_df)} 条样本
- 验证集: {len(trial_aspect_df)} 条样本
- 类别数: {len(ASPECT_CATEGORIES)}
- 类别: {', '.join(ASPECT_CATEGORIES)}

### Pipeline 2: 方面情感分析（4分类）
- 训练集: {len(train_sentiment_df)} 条样本
- 验证集: {len(trial_sentiment_df)} 条样本
- 情感标签: {', '.join(SENTIMENT_LABELS)}

## 输出文件列表

### Pipeline 1 数据集
- train_aspect_detection.json / .csv
- trial_aspect_detection.json / .csv

### Pipeline 2 数据集
- train_sentiment.json / .csv
- trial_sentiment.json / .csv

### 原始数据
- train_data_raw.json
- trial_data_raw.json
- test_phase_a_data.json
- test_phase_b_data.json
"""

print(summary)

# 保存摘要
with open('GroupXX_Dataset_files/DATASET_SUMMARY.md', 'w', encoding='utf-8') as f:
    f.write(summary)

print("\n✓ 数据预处理完成！")